In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

In [2]:
x_raw = np.array([
    [1, 2, 3, 4, 2],  # checked in, various activities -> Occupied (1)
    [1, 4, 3, 2, 4],  # checked in, various activities -> Occupied (1)
    [1, 2, 5, 0, 0],  # checked in, then checked out -> Vacant (0)
    [1, 3, 4, 2, 5],  # checked in, activities, then checked out -> Vacant (0)
    [0, 0, 0, 0, 0],  # never checked in -> Vacant (0)
    [1, 2, 2, 3, 5],  # checked out at the end -> Vacant (0)
    [1, 3, 2, 4, 3],  # still occupied -> Occupied (1)
    [1, 5, 0, 0, 0],  # immediate checkout -> Vacant (0)
],dtype=np.int32)

y = np.array([1, 1, 0, 0, 0, 0, 1, 0],dtype=np.float32)

In [3]:
# 2. Build a small, clean LSTM model
vocab_size = 6 # Tokens from 0 to 5
sequence_len = 5

In [4]:
model = tf.keras.Sequential([
    
    tf.keras.layers.Input(shape=(sequence_len,)),

    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=4, mask_zero=True),

    tf.keras.layers.LSTM(units=8, return_sequences=False),

    tf.keras.layers.Dense(units=1, activation='sigmoid')
])

In [5]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [6]:
hsitory = model.fit(
    x_raw,
    y,
    epochs=100,
    verbose=1
)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 1.0000 - loss: 0.6913
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.6250 - loss: 0.6871
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.6250 - loss: 0.6825
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.6250 - loss: 0.6773
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.6250 - loss: 0.6715
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.6250 - loss: 0.6649
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.6250 - loss: 0.6573
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.6250 - loss: 0.6487
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.6250 - loss: 0.6389
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.6250 - loss: 0.6280
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.6250 - loss: 0.6159
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.7500 - loss

In [7]:
test_cases = np.array([
    [1, 4, 4, 3, 2],  # Should be 1 (Occupied)
    [1, 4, 4, 3, 5],  # Should be 0 (Vacant)
], dtype=np.int32)

prdictions = model.predict(test_cases)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 303ms/step


In [8]:
for seq, score in zip(test_cases, prdictions):
    status = "Occupied" if score[0] > 0.5 else "Vacant"
    print(f"Sequence {seq} -> Score: {score[0]:.4f} ({status})")

Sequence [1 4 4 3 2] -> Score: 0.9932 (Occupied)
Sequence [1 4 4 3 5] -> Score: 0.0055 (Vacant)


In [9]:
print(prdictions)

[[0.99321467]
 [0.0054879 ]]


In [10]:
# Create an identical LSTM layer to extract states
lstm_inspector = tf.keras.layers.LSTM(units=8, return_state=True)

# Pass test_cases through the trained embedding layer first
embedded_tests = model.layers[0](test_cases)

# Run through the inspector to get: output, final_hidden_state (h), and final_cell_state (c)
whole_seq_output, final_h, final_c = lstm_inspector(embedded_tests)

print("Occupied case - Cell State (c) vector norm:", np.linalg.norm(final_c[0]))
print("Vacant case   - Cell State (c) vector norm:", np.linalg.norm(final_c[1]))

Occupied case - Cell State (c) vector norm: 0.3454433
Vacant case   - Cell State (c) vector norm: 0.261401
